# Filter tomato polygons (keep all attributes)

**2018 folder:** Use this notebook after [`01_2018_explore_shapefile_and_tomato_codes.ipynb`](01_2018_explore_shapefile_and_tomato_codes.ipynb). For **other survey years**, change `landiq.year` (and related fields) in `configs/paths.local.yaml` first — same notebook pattern works for any year.

**Behavior:** Keeps only **rows** that match your tomato codes. **Every attribute column** (County, Acres, `CROPTYP*`, year-specific `Crop…` text fields, …) and **geometry** stay on those rows.

Configure **`configs/paths.local.yaml`**: `landiq.year`, `landiq.tomato_values`, **`landiq.crop_columns`** (or `crop_column`), optional **`output_filename`**. Uses `REPO_ROOT` and `landiq_tomato_gpkg_path()`.

**Shapefile location:** By default the notebook looks for a single `*.shp` under `data/raw/landiq/<year>/`. If you extracted to **`data/derived/landiq_extracted/…`**, set **`landiq.input_shapefile`** in `paths.local.yaml` to that repo-relative `.shp` path.

**CLI:** `python -m src.landiq.filter_tomato` (optional `--input` overrides config).

In [ ]:
import geopandas as gpd
import yaml

from src.landiq.filter_tomato import filter_tomatoes_from_landiq_config
from src.utils.paths import REPO_ROOT, landiq_tomato_gpkg_path, resolve_landiq_shapefile_path

cfg_path = REPO_ROOT / "configs" / "paths.local.yaml"
if not cfg_path.is_file():
    cfg_path = REPO_ROOT / "configs" / "paths.example.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
lc = cfg["landiq"]

in_path = resolve_landiq_shapefile_path(cfg, REPO_ROOT)
gdf = gpd.read_file(in_path)
tomato = filter_tomatoes_from_landiq_config(gdf, lc)

out_gpkg = landiq_tomato_gpkg_path(cfg, REPO_ROOT)
out_gpkg.parent.mkdir(parents=True, exist_ok=True)
tomato.to_file(out_gpkg, driver="GPKG")

assert list(tomato.columns) == list(gdf.columns), "Schema should match (row filter only)"
print(f"Rows: {len(gdf)} -> {len(tomato)} tomato | columns: {len(tomato.columns)} (all attributes kept)")
print("Wrote:", out_gpkg.resolve())

In [ ]:
# Inspect filtered output (expects gdf, tomato, lc from the cell above)
from IPython.display import display

print("Shapes — input:", gdf.shape, "| tomato:", tomato.shape)
print("CRS:", tomato.crs)
attr = [c for c in tomato.columns if c != "geometry"]
print(f"Columns ({len(tomato.columns)}):", attr)

print("\n--- head(5) — attributes only ---")
display(tomato[attr].head())
print("--- tail(3) ---")
display(tomato[attr].tail(3))

_cc = lc.get("crop_columns") or ([lc["crop_column"]] if lc.get("crop_column") else [])
for col in _cc:
    if col in tomato.columns:
        print(f"\n--- {col} value_counts (top 10) ---")
        display(tomato[col].value_counts().head(10))